# Chicken Threat Detector Data v3

This notebook is the **data-collection and dataset-build stage** for the chicken threat detector. It keeps the v2 data path intact and stops after creating, validating, visualizing, balancing, and exporting the YOLO dataset.

Run this notebook first. It produces `chicken_threat_dataset_v3.zip`, which is the input for `ChickenThreadDetectorModelv3.ipynb`.

Pipeline boundary:

- Download / import raw YOLO sources.
- Convert polygon labels to detection boxes where needed.
- Map source classes into the canonical detector classes.
- Merge poultry + predator sources.
- Balance the final dataset with predator targets and reduced person count.
- Validate and visually inspect the final YOLO dataset.
- Export the final dataset as a ZIP.


## Install dependencies

Run this once per notebook runtime. In Colab, dependency installation can take several minutes, especially for FiftyOne. If Colab asks to restart the runtime after installing packages, restart and rerun from this cell.


In [ ]:
# This cell handles dependencies.
# If you see an ImportError after running this, please Restart the Session (Runtime > Restart session).
%pip install -q --upgrade pyyaml matplotlib tqdm roboflow fiftyone "pillow>=10.4.0,<11.0.0" ultralytics

## Colab runtime setup

By default, Colab stores generated datasets under `/content/chicken_threat_detector`, which disappears when the runtime resets. Set `USE_GOOGLE_DRIVE = True` to store data under your Drive instead.


In [ ]:
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in str(get_ipython())
USE_GOOGLE_DRIVE = False

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ.setdefault(
        "CHICKEN_THREAT_WORKDIR",
        "/content/drive/MyDrive/chicken_threat_detector",
    )
elif IN_COLAB:
    os.environ.setdefault("CHICKEN_THREAT_WORKDIR", "/content/chicken_threat_detector")

print(f"IN_COLAB={IN_COLAB}")
print(f"CHICKEN_THREAT_WORKDIR={os.getenv('CHICKEN_THREAT_WORKDIR', 'data/chicken_threat_detector')}")


## Configuration

Set `ROBOFLOW_API_KEY` either as an environment variable or as a Colab Secret. The notebook does not store API keys. Leave download flags disabled when using datasets that were already uploaded or saved in Drive.


In [ ]:
import os
import re
import shutil
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

SPLITS = ("train", "valid", "test")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DEFAULT_WORK_DIR = "/content/chicken_threat_detector" if "google.colab" in sys.modules else "data/chicken_threat_detector"
WORK_DIR = Path(os.getenv("CHICKEN_THREAT_WORKDIR", DEFAULT_WORK_DIR)).resolve()
RAW_ROOT = WORK_DIR / "raw"
PROCESSED_ROOT = WORK_DIR / "processed"
FINAL_DATASET_ROOT = WORK_DIR / "dataset_yolo"

CANONICAL_NAMES = [
    "chicken",
    "other_poultry",
    "rodent",
    "fox",
    "cat",
    "dog",
    "marten_weasel",
    "bird_of_prey",
    "other_bird",
    "person",
]
CANONICAL_ID = {name: index for index, name in enumerate(CANONICAL_NAMES)}

POULTRY_CLASS_MAP = {
    "Chicken": "chicken",
    "Duck": "other_poultry",
    "Geese": "other_poultry",
    "Goose": "other_poultry",
    "Turkey": "other_poultry",
    "Rodent": "rodent",
    "Ostrich": None,
    "Quail": None,
    "Emu": None,
    "Guinea Fowl": None,
    "Egg": None,
}

OPEN_IMAGES_CLASS_MAP = {
    "Fox": "fox",
    "Cat": "cat",
    "Dog": "dog",
    "Eagle": "bird_of_prey",
    "Owl": "bird_of_prey",
    "Bird": "other_bird",
    "Chicken": "chicken",
    "Duck": "other_poultry",
    "Goose": "other_poultry",
    # Keep Person low. It is context/hard-negative material, not a target threat class.
    "Person": "person",
}

ROBOFLOW_WORKSPACE = "tmp-dulho"
ROBOFLOW_PROJECT = "poultry-and-predators-detection-um2x6"
ROBOFLOW_VERSION = 2
ROBOFLOW_RAW_ROOT = RAW_ROOT / "poultry-and-predators-detection-2"

# Set these to True only when you want this notebook to download data into WORK_DIR.
DOWNLOAD_ROBOFLOW = True
DOWNLOAD_OPEN_IMAGES = True
# Class-balanced Open Images download targets.
# This replaces a single global max_samples value, because global sampling tends to over-collect Person.
OPEN_IMAGES_TARGET_IMAGES_BY_SOURCE_CLASS = {
    "train": {
        "Fox": 700,
        "Cat": 700,
        "Dog": 700,
        "Eagle": 500,
        "Owl": 500,
        "Bird": 500,
        "Chicken": 250,
        "Duck": 250,
        "Goose": 250,
        "Person": 250,
    },
    "valid": {
        "Fox": 120,
        "Cat": 120,
        "Dog": 120,
        "Eagle": 100,
        "Owl": 100,
        "Bird": 100,
        "Chicken": 60,
        "Duck": 60,
        "Goose": 60,
        "Person": 60,
    },
    "test": {
        "Fox": 120,
        "Cat": 120,
        "Dog": 120,
        "Eagle": 100,
        "Owl": 100,
        "Bird": 100,
        "Chicken": 60,
        "Duck": 60,
        "Goose": 60,
        "Person": 60,
    },
}

# Check each additional YOLO source maps source class names to canonical names or None.
ADDITIONAL_YOLO_SOURCES = []

for directory in (RAW_ROOT, PROCESSED_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

WORK_DIR

## Shared YOLO helpers

In [ ]:
@dataclass(frozen=True)
class ProcessStats:
    source: str
    images_seen: int = 0
    images_written: int = 0
    labels_seen: int = 0
    labels_written: int = 0
    empty_label_images: int = 0
    missing_label_images: int = 0
    dropped_labels: int = 0
    invalid_labels: int = 0


def clean_dir(path: Path) -> None:
    path = path.resolve()
    allowed_roots = [WORK_DIR.resolve(), PROCESSED_ROOT.resolve(), FINAL_DATASET_ROOT.resolve()]
    if not any(path == root or root in path.parents for root in allowed_roots):
        raise ValueError(f"Refusing to clean path outside the notebook work directory: {path}")
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def slugify(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value.strip())
    return value.strip("._") or "item"


def load_class_names(dataset_root: Path) -> list[str]:
    data_yaml = dataset_root / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(f"Missing data.yaml in {dataset_root}")

    data = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
    names = data.get("names")
    if isinstance(names, dict):
        return [str(names[key]) for key in sorted(names, key=lambda key: int(key))]
    if isinstance(names, list):
        return [str(name) for name in names]
    raise ValueError(f"Unsupported names format in {data_yaml}")


def write_data_yaml(dataset_root: Path, names: list[str] = CANONICAL_NAMES) -> None:
    data = {
        "path": str(dataset_root.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(names),
        "names": names,
    }
    (dataset_root / "data.yaml").write_text(yaml.safe_dump(data, sort_keys=False), encoding="utf-8")


def image_files(image_dir: Path) -> list[Path]:
    if not image_dir.exists():
        return []
    return sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)


def copy_image(source: Path, dest_dir: Path, dest_stem: str) -> Path:
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / f"{dest_stem}{source.suffix.lower()}"
    shutil.copy2(source, dest)
    return dest


def polygon_to_box(coords: list[float]) -> tuple[float, float, float, float] | None:
    if len(coords) < 6 or len(coords) % 2 != 0:
        return None
    xs = coords[0::2]
    ys = coords[1::2]
    x_min, x_max = max(0.0, min(xs)), min(1.0, max(xs))
    y_min, y_max = max(0.0, min(ys)), min(1.0, max(ys))
    width = x_max - x_min
    height = y_max - y_min
    if width <= 0 or height <= 0:
        return None
    return ((x_min + x_max) / 2, (y_min + y_max) / 2, width, height)


def clip_yolo_box(box: Iterable[float]) -> tuple[float, float, float, float] | None:
    x_center, y_center, width, height = [float(value) for value in box]
    if width <= 0 or height <= 0:
        return None

    x_min = max(0.0, x_center - width / 2)
    y_min = max(0.0, y_center - height / 2)
    x_max = min(1.0, x_center + width / 2)
    y_max = min(1.0, y_center + height / 2)
    width = x_max - x_min
    height = y_max - y_min
    if width <= 0 or height <= 0:
        return None
    return ((x_min + x_max) / 2, (y_min + y_max) / 2, width, height)


def parse_yolo_line(line: str) -> tuple[int, tuple[float, float, float, float]] | None:
    parts = line.split()
    if len(parts) < 5:
        return None
    try:
        class_id = int(float(parts[0]))
        coords = [float(value) for value in parts[1:]]
    except ValueError:
        return None

    if len(coords) == 4:
        box = clip_yolo_box(coords)
    else:
        box = polygon_to_box(coords)
    if box is None:
        return None
    return class_id, box


def format_yolo_line(class_id: int, box: tuple[float, float, float, float]) -> str:
    return f"{class_id} " + " ".join(f"{value:.6f}" for value in box)


## Optional: upload an existing dataset zip in Colab

Use this when a collaborator has a YOLO export zip instead of using the Roboflow API. Upload one zip, then set the extracted path as `ROBOFLOW_RAW_ROOT` or add it to `ADDITIONAL_YOLO_SOURCES`.


In [ ]:
def upload_and_extract_zip(destination: Path) -> Path:
    if not IN_COLAB:
        raise RuntimeError("This helper is intended for Google Colab uploads.")

    from google.colab import files
    import zipfile

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one zip file.")

    zip_name = next(iter(uploaded))
    zip_path = Path(zip_name).resolve()
    clean_dir(destination)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(destination)

    nested = [path for path in destination.iterdir() if path.is_dir() and (path / "data.yaml").exists()]
    extracted_root = nested[0] if nested else destination
    print(f"Extracted dataset root: {extracted_root}")
    return extracted_root


# Example:
#ROBOFLOW_RAW_ROOT = upload_and_extract_zip(RAW_ROOT / "poultry-and-predators-detection-2")


## Inspect source datasets

In [ ]:
def summarize_yolo_source(dataset_root: Path) -> dict[str, Counter]:
    class_names = load_class_names(dataset_root)
    summary: dict[str, Counter] = {}

    for split in SPLITS:
        label_dir = dataset_root / split / "labels"
        image_dir = dataset_root / split / "images"
        counter: Counter = Counter()
        counter["__images__"] = len(image_files(image_dir))
        counter["__empty_label_files__"] = 0
        counter["__missing_label_files__"] = 0

        for image_path in image_files(image_dir):
            label_path = label_dir / f"{image_path.stem}.txt"
            if not label_path.exists():
                counter["__missing_label_files__"] += 1
                continue
            text = label_path.read_text(encoding="utf-8").strip()
            if not text:
                counter["__empty_label_files__"] += 1
                continue
            for line in text.splitlines():
                parsed = parse_yolo_line(line)
                if parsed is None:
                    counter["__invalid_labels__"] += 1
                    continue
                class_id, _ = parsed
                class_name = class_names[class_id] if 0 <= class_id < len(class_names) else f"unknown:{class_id}"
                counter[class_name] += 1

        summary[split] = counter
    return summary


def print_summary(summary: dict[str, Counter]) -> None:
    for split, counter in summary.items():
        object_total = sum(count for label, count in counter.items() if not label.startswith("__"))
        print(f"{split.upper()} ({object_total} objects, {counter['__images__']} images)")
        for label, count in counter.most_common():
            if label.startswith("__"):
                continue
            print(f"  - {label}: {count}")
        print(f"  empty label files: {counter['__empty_label_files__']}")
        print(f"  missing label files: {counter['__missing_label_files__']}")
        if counter["__invalid_labels__"]:
            print(f"  invalid labels: {counter['__invalid_labels__']}")
        print()


if ROBOFLOW_RAW_ROOT.exists():
    print_summary(summarize_yolo_source(ROBOFLOW_RAW_ROOT))
else:
    print(f"Roboflow raw dataset not found yet: {ROBOFLOW_RAW_ROOT}")


## Download the Roboflow poultry dataset

This downloads the close poultry seed dataset in YOLOv8 format. Keep this disabled if you already downloaded/exported it manually into `ROBOFLOW_RAW_ROOT`.


In [ ]:
import traceback
import time

def get_secret(name: str) -> str | None:
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

def download_roboflow_dataset(destination: Path = ROBOFLOW_RAW_ROOT) -> Path:
    from roboflow import Roboflow

    api_key = get_secret("ROBOFLOW_API_KEY")
    if not api_key:
        raise RuntimeError("Set ROBOFLOW_API_KEY as an environment variable or Colab Secret before running the Roboflow download.")

    print(f"Attempting to download {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION}...")

    try:
        rf = Roboflow(api_key=api_key)
        project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
        version = project.version(ROBOFLOW_VERSION)

        # Check if version exists and is generated
        print(f"Dataset version state: {getattr(version, 'state', 'unknown')}")

        # Clean directory just before download to ensure a fresh state
        clean_dir(destination)

        # Force download yolov8 format
        dataset = version.download("yolov8", location=str(destination), overwrite=True)

        extracted_path = Path(dataset.location).resolve()

        # Immediate verification
        contents = list(extracted_path.iterdir())
        print(f"Download reported success. Local folder contains {len(contents)} items.")

        if len(contents) == 0:
            print("WARNING: Download returned an empty directory. This often happens if the export is still generating on Roboflow.")
            print("Wait 30 seconds and try running this cell again.")

        yaml_path = extracted_path / "data.yaml"
        if not yaml_path.exists():
            # Check for nested directory (sometimes Roboflow zips have an extra level)
            nested = [p for p in extracted_path.iterdir() if p.is_dir() and (p / "data.yaml").exists()]
            if nested:
                print(f"Found data.yaml in nested folder: {nested[0]}")
                return nested[0]
            print(f"CRITICAL: data.yaml missing in {extracted_path}!")

        return extracted_path
    except Exception as e:
        print(f"An error occurred during Roboflow download:")
        traceback.print_exc()
        raise e

if DOWNLOAD_ROBOFLOW:
    try:
        ROBOFLOW_RAW_ROOT = download_roboflow_dataset()
        print(f"Active Roboflow root: {ROBOFLOW_RAW_ROOT}")
    except Exception as e:
        print("Download failed. Please check your API key and Internet connection.")
else:
    print("Skipping Roboflow download. Set DOWNLOAD_ROBOFLOW = True to run it.")

## Convert and filter YOLO sources

`process_yolo_source` converts segmentation polygons to detection boxes, drops unwanted labels, rewrites class IDs into the canonical class map, and copies only usable images into a processed source folder.


In [ ]:
def process_yolo_source(
    source_root: Path,
    output_root: Path,
    class_map: dict[str, str | None],
    source_name: str,
    keep_empty_images: bool = False,
) -> ProcessStats:
    source_root = Path(source_root)
    output_root = Path(output_root)
    class_names = load_class_names(source_root)
    clean_dir(output_root)

    totals = defaultdict(int)

    for split in SPLITS:
        src_image_dir = source_root / split / "images"
        src_label_dir = source_root / split / "labels"
        dst_image_dir = output_root / split / "images"
        dst_label_dir = output_root / split / "labels"
        dst_image_dir.mkdir(parents=True, exist_ok=True)
        dst_label_dir.mkdir(parents=True, exist_ok=True)

        for image_path in tqdm(image_files(src_image_dir), desc=f"{source_name}:{split}"):
            totals["images_seen"] += 1
            label_path = src_label_dir / f"{image_path.stem}.txt"
            if not label_path.exists():
                totals["missing_label_images"] += 1
                if not keep_empty_images:
                    continue
                lines = []
            else:
                text = label_path.read_text(encoding="utf-8").strip()
                if not text:
                    totals["empty_label_images"] += 1
                    if not keep_empty_images:
                        continue
                    lines = []
                else:
                    lines = text.splitlines()

            output_lines: list[str] = []
            for line in lines:
                parsed = parse_yolo_line(line)
                if parsed is None:
                    totals["invalid_labels"] += 1
                    continue
                old_class_id, box = parsed
                totals["labels_seen"] += 1
                if not 0 <= old_class_id < len(class_names):
                    totals["invalid_labels"] += 1
                    continue

                source_class = class_names[old_class_id]
                canonical_class = class_map.get(source_class)
                if canonical_class is None:
                    totals["dropped_labels"] += 1
                    continue
                if canonical_class not in CANONICAL_ID:
                    raise ValueError(f"Unknown canonical class {canonical_class!r} for {source_class!r}")

                output_lines.append(format_yolo_line(CANONICAL_ID[canonical_class], box))

            if not output_lines and not keep_empty_images:
                continue

            dest_stem = f"{slugify(source_name)}_{slugify(image_path.stem)}"
            copy_image(image_path, dst_image_dir, dest_stem)
            (dst_label_dir / f"{dest_stem}.txt").write_text("\n".join(output_lines) + ("\n" if output_lines else ""), encoding="utf-8")
            totals["images_written"] += 1
            totals["labels_written"] += len(output_lines)

    write_data_yaml(output_root)
    return ProcessStats(source=source_name, **totals)


def process_poultry_dataset() -> Path | None:
    if not ROBOFLOW_RAW_ROOT.exists():
        print(f"Poultry raw dataset not found: {ROBOFLOW_RAW_ROOT}")
        return None

    output_root = PROCESSED_ROOT / "poultry"
    stats = process_yolo_source(
        source_root=ROBOFLOW_RAW_ROOT,
        output_root=output_root,
        class_map=POULTRY_CLASS_MAP,
        source_name="poultry",
        keep_empty_images=False,
    )
    print(stats)
    print_summary(summarize_yolo_source(output_root))
    return output_root


poultry_processed_root = process_poultry_dataset()


## Download and export balanced Open Images predator/context samples

Open Images is downloaded per source class instead of with one global `max_samples`.
This avoids collecting thousands of `Person` instances while still giving fox, cat, dog, and bird-of-prey enough samples.


In [ ]:
def find_fiftyone_detections_field(dataset) -> str:
    for field_name, field in dataset.get_field_schema().items():
        document_type = getattr(field, "document_type", None)
        if document_type is not None and document_type.__name__ == "Detections":
            return field_name
    raise ValueError("Could not find a FiftyOne Detections field in the dataset")


def export_fiftyone_split_to_yolo(
    dataset,
    output_root: Path,
    split: str,
    class_map: dict[str, str | None],
    source_name: str,
) -> ProcessStats:
    detection_field = find_fiftyone_detections_field(dataset)
    dst_image_dir = output_root / split / "images"
    dst_label_dir = output_root / split / "labels"
    dst_image_dir.mkdir(parents=True, exist_ok=True)
    dst_label_dir.mkdir(parents=True, exist_ok=True)

    totals = defaultdict(int)

    for sample in tqdm(dataset, desc=f"{source_name}:{split}"):
        totals["images_seen"] += 1
        detections = sample[detection_field].detections if sample[detection_field] is not None else []
        output_lines: list[str] = []

        for detection in detections:
            totals["labels_seen"] += 1
            canonical_class = class_map.get(detection.label)
            if canonical_class is None:
                totals["dropped_labels"] += 1
                continue
            if canonical_class not in CANONICAL_ID:
                raise ValueError(f"Unknown canonical class {canonical_class!r} for {detection.label!r}")

            x_min, y_min, width, height = [float(value) for value in detection.bounding_box]
            box = clip_yolo_box((x_min + width / 2, y_min + height / 2, width, height))
            if box is None:
                totals["invalid_labels"] += 1
                continue
            output_lines.append(format_yolo_line(CANONICAL_ID[canonical_class], box))

        if not output_lines:
            continue

        filepath = Path(sample.filepath)
        dest_stem = f"{slugify(source_name)}_{slugify(split)}_{slugify(sample.id)}"
        copy_image(filepath, dst_image_dir, dest_stem)
        (dst_label_dir / f"{dest_stem}.txt").write_text("\n".join(output_lines) + "\n", encoding="utf-8")
        totals["images_written"] += 1
        totals["labels_written"] += len(output_lines)

    return ProcessStats(source=f"{source_name}:{split}", **totals)


def build_open_images_source(output_root: Path = PROCESSED_ROOT / "open_images_balanced") -> Path:
    import fiftyone.zoo as foz

    clean_dir(output_root)
    stats: list[ProcessStats] = []
    split_map = {"train": "train", "valid": "validation", "test": "test"}

    for canonical_split, open_images_split in split_map.items():
        targets = OPEN_IMAGES_TARGET_IMAGES_BY_SOURCE_CLASS[canonical_split]

        for source_class, max_samples in targets.items():
            if max_samples <= 0:
                continue
            if source_class not in OPEN_IMAGES_CLASS_MAP:
                raise ValueError(f"{source_class!r} has no OPEN_IMAGES_CLASS_MAP entry")

            source_map = {source_class: OPEN_IMAGES_CLASS_MAP[source_class]}
            dataset_name = f"chicken-threat-open-images-{open_images_split}-{slugify(source_class)}"

            dataset = foz.load_zoo_dataset(
                "open-images-v7",
                split=open_images_split,
                label_types=["detections"],
                classes=[source_class],
                max_samples=max_samples,
                shuffle=True,
                seed=42,
                # Only export the target class for this shard. This prevents Person from leaking
                # into every animal subset and dominating the final distribution.
                only_matching=True,
                dataset_name=dataset_name,
                overwrite=True,
            )
            stats.append(
                export_fiftyone_split_to_yolo(
                    dataset=dataset,
                    output_root=output_root,
                    split=canonical_split,
                    class_map=source_map,
                    source_name=f"open_images_{source_class}",
                )
            )

    write_data_yaml(output_root)
    for item in stats:
        print(item)
    print_summary(summarize_yolo_source(output_root))
    return output_root


if DOWNLOAD_OPEN_IMAGES:
    open_images_processed_root = build_open_images_source()
else:
    open_images_processed_root = PROCESSED_ROOT / "open_images_balanced" if (PROCESSED_ROOT / "open_images_balanced").exists() else None
    print("Skipping Open Images download. Set DOWNLOAD_OPEN_IMAGES = True to run it.")


## Process additional YOLO predator datasets

Use this for manually downloaded Roboflow or camera-trap YOLO exports. Each source must provide a `data.yaml`, split image folders, split label folders, and a class-name mapping into the canonical schema.


In [ ]:
additional_processed_roots: list[Path] = []

for source in ADDITIONAL_YOLO_SOURCES:
    source_root = Path(source["root"])
    source_name = str(source["name"])
    if not source_root.exists():
        print(f"Skipping missing source {source_name}: {source_root}")
        continue

    output_root = PROCESSED_ROOT / slugify(source_name)
    stats = process_yolo_source(
        source_root=source_root,
        output_root=output_root,
        class_map=source["class_map"],
        source_name=source_name,
        keep_empty_images=bool(source.get("keep_empty_images", False)),
    )
    additional_processed_roots.append(output_root)
    print(stats)
    print_summary(summarize_yolo_source(output_root))


## Merge processed sources

In [ ]:
def merge_canonical_yolo_sources(source_roots: list[Path], output_root: Path = FINAL_DATASET_ROOT) -> Path:
    clean_dir(output_root)

    for split in SPLITS:
        (output_root / split / "images").mkdir(parents=True, exist_ok=True)
        (output_root / split / "labels").mkdir(parents=True, exist_ok=True)

    for source_root in source_roots:
        source_prefix = slugify(source_root.name)
        for split in SPLITS:
            src_image_dir = source_root / split / "images"
            src_label_dir = source_root / split / "labels"
            dst_image_dir = output_root / split / "images"
            dst_label_dir = output_root / split / "labels"

            for image_path in image_files(src_image_dir):
                label_path = src_label_dir / f"{image_path.stem}.txt"
                if not label_path.exists():
                    continue
                dest_stem = f"{source_prefix}_{slugify(image_path.stem)}"
                copy_image(image_path, dst_image_dir, dest_stem)
                shutil.copy2(label_path, dst_label_dir / f"{dest_stem}.txt")

    write_data_yaml(output_root)
    return output_root


processed_roots = [root for root in [poultry_processed_root, open_images_processed_root, *additional_processed_roots] if root is not None and Path(root).exists()]

if processed_roots:
    final_dataset_root = merge_canonical_yolo_sources([Path(root) for root in processed_roots])
    print(f"Final dataset written to {final_dataset_root}")
    print_summary(summarize_yolo_source(final_dataset_root))
else:
    final_dataset_root = None
    print("No processed sources are available yet.")


## Balance final dataset distribution

This down-samples overrepresented classes after merging all sources. It intentionally caps `person`
and keeps predator classes near the same training count.


In [ ]:
BALANCED_DATASET_ROOT = WORK_DIR / "dataset_yolo_balanced"

# Object-count targets, not image-count targets.
# Predator train targets are intentionally >= 500 objects each.
TARGET_OBJECTS_BY_SPLIT = {
    "train": {
        "chicken": 700,
        "other_poultry": 900,
        "rodent": 500,
        "fox": 600,
        "cat": 600,
        "dog": 600,
        "marten_weasel": 600,
        "bird_of_prey": 600,
        "other_bird": 600,
        "person": 300,
    },
    "valid": {
        "chicken": 120,
        "other_poultry": 150,
        "rodent": 90,
        "fox": 100,
        "cat": 100,
        "dog": 100,
        "marten_weasel": 100,
        "bird_of_prey": 100,
        "other_bird": 100,
        "person": 60,
    },
    "test": {
        "chicken": 120,
        "other_poultry": 150,
        "rodent": 90,
        "fox": 100,
        "cat": 100,
        "dog": 100,
        "marten_weasel": 100,
        "bird_of_prey": 100,
        "other_bird": 100,
        "person": 60,
    },
}

MAX_OBJECTS_BY_SPLIT = {
    split: {
        name: int(target * (1.15 if name != "person" else 1.0))
        for name, target in targets.items()
    }
    for split, targets in TARGET_OBJECTS_BY_SPLIT.items()
}


def label_counts_for_file(label_path: Path) -> Counter:
    counts: Counter = Counter()
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parsed = parse_yolo_line(line)
        if parsed is None:
            continue
        class_id, _box = parsed
        if 0 <= class_id < len(CANONICAL_NAMES):
            counts[CANONICAL_NAMES[class_id]] += 1
    return counts


def split_records(dataset_root: Path, split: str) -> list[dict]:
    image_dir = dataset_root / split / "images"
    label_dir = dataset_root / split / "labels"
    records = []

    for image_path in image_files(image_dir):
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            continue
        counts = label_counts_for_file(label_path)
        if not counts:
            continue
        records.append({"image_path": image_path, "label_path": label_path, "counts": counts})

    # Prefer scarce predators, then images without persons, then simple single-object frames.
    scarce_order = {"marten_weasel": 0, "bird_of_prey": 1, "fox": 2, "rodent": 3, "cat": 4, "dog": 5}
    def sort_key(record: dict) -> tuple:
        counts = record["counts"]
        scarce_score = min((scarce_order[name] for name in counts if name in scarce_order), default=99)
        return (scarce_score, counts.get("person", 0), sum(counts.values()))

    return sorted(records, key=sort_key)


def can_add(current: Counter, add: Counter, caps: dict[str, int]) -> bool:
    for name, value in add.items():
        if name in caps and current[name] + value > caps[name]:
            return False
    return True


def improves_target(current: Counter, add: Counter, targets: dict[str, int]) -> bool:
    return any(add.get(name, 0) and current[name] < target for name, target in targets.items())


def copy_record(record: dict, dst_image_dir: Path, dst_label_dir: Path, prefix: str) -> None:
    dest_stem = f"{prefix}_{slugify(record['image_path'].stem)}"
    copy_image(record["image_path"], dst_image_dir, dest_stem)
    shutil.copy2(record["label_path"], dst_label_dir / f"{dest_stem}.txt")


def balance_canonical_yolo_dataset(input_root: Path, output_root: Path = BALANCED_DATASET_ROOT) -> Path:
    clean_dir(output_root)

    for split in SPLITS:
        dst_image_dir = output_root / split / "images"
        dst_label_dir = output_root / split / "labels"
        dst_image_dir.mkdir(parents=True, exist_ok=True)
        dst_label_dir.mkdir(parents=True, exist_ok=True)

        targets = TARGET_OBJECTS_BY_SPLIT[split]
        caps = MAX_OBJECTS_BY_SPLIT[split]
        records = split_records(input_root, split)
        selected: list[dict] = []
        counts: Counter = Counter()

        # Pass 1: fill underrepresented targets without exceeding caps.
        for record in records:
            if improves_target(counts, record["counts"], targets) and can_add(counts, record["counts"], caps):
                selected.append(record)
                counts.update(record["counts"])

        # Pass 2: add remaining images only when they do not break caps and still improve balance.
        for record in records:
            if record in selected:
                continue
            if can_add(counts, record["counts"], caps) and improves_target(counts, record["counts"], targets):
                selected.append(record)
                counts.update(record["counts"])

        for index, record in enumerate(selected):
            copy_record(record, dst_image_dir, dst_label_dir, prefix=f"{split}_{index:06d}")

        print(f"\nBalanced {split}: selected {len(selected)} / {len(records)} images")
        for name in CANONICAL_NAMES:
            target = targets.get(name, 0)
            status = "OK" if counts[name] >= min(target, counts[name]) else "CHECK"
            deficit = max(0, target - counts[name])
            suffix = f" deficit={deficit}" if deficit else ""
            print(f"  {name:15s} {counts[name]:5d} target={target:5d}{suffix}")

    write_data_yaml(output_root)
    return output_root


if final_dataset_root is not None:
    unbalanced_dataset_root = final_dataset_root
    final_dataset_root = balance_canonical_yolo_dataset(final_dataset_root)
    print(f"\nBalanced dataset written to {final_dataset_root}")
    print_summary(summarize_yolo_source(final_dataset_root))
else:
    print("No final dataset to balance yet.")


## Validate final YOLO dataset

In [ ]:
def validate_yolo_dataset(dataset_root: Path) -> list[str]:
    errors: list[str] = []
    names = load_class_names(dataset_root)
    if names != CANONICAL_NAMES:
        errors.append("data.yaml names do not match CANONICAL_NAMES")

    for split in SPLITS:
        image_dir = dataset_root / split / "images"
        label_dir = dataset_root / split / "labels"
        image_stems = {path.stem for path in image_files(image_dir)}
        label_stems = {path.stem for path in label_dir.glob("*.txt")} if label_dir.exists() else set()

        missing_labels = image_stems - label_stems
        orphan_labels = label_stems - image_stems
        if missing_labels:
            errors.append(f"{split}: {len(missing_labels)} images have no label file")
        if orphan_labels:
            errors.append(f"{split}: {len(orphan_labels)} label files have no image")

        for label_path in sorted(label_dir.glob("*.txt")) if label_dir.exists() else []:
            for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
                if not line.strip():
                    continue
                parsed = parse_yolo_line(line)
                if parsed is None:
                    errors.append(f"{label_path}:{line_number}: invalid YOLO row")
                    continue
                class_id, box = parsed
                if not 0 <= class_id < len(CANONICAL_NAMES):
                    errors.append(f"{label_path}:{line_number}: class id {class_id} out of range")
                if any(value < 0 or value > 1 for value in box):
                    errors.append(f"{label_path}:{line_number}: box value outside 0..1")

    return errors


if final_dataset_root is not None:
    validation_errors = validate_yolo_dataset(final_dataset_root)
    if validation_errors:
        print("Validation failed:")
        for error in validation_errors[:50]:
            print(f"  - {error}")
        if len(validation_errors) > 50:
            print(f"  ... {len(validation_errors) - 50} more errors")
    else:
        print("Validation passed.")
else:
    print("No final dataset to validate yet.")


## Visual quality checks

Run this after creating `final_dataset_root`. Inspect samples before training. In particular, verify that predators are labeled where visible, irrelevant large-bird labels are gone, and poultry polygons became reasonable boxes.


In [ ]:
def draw_yolo_boxes(image_path: Path, label_path: Path, names: list[str]) -> Image.Image:
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    draw = ImageDraw.Draw(image)

    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parsed = parse_yolo_line(line)
            if parsed is None:
                continue
            class_id, (x_center, y_center, box_width, box_height) = parsed
            x_min = (x_center - box_width / 2) * width
            y_min = (y_center - box_height / 2) * height
            x_max = (x_center + box_width / 2) * width
            y_max = (y_center + box_height / 2) * height
            label = names[class_id] if 0 <= class_id < len(names) else str(class_id)
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=3)
            draw.text((x_min + 3, y_min + 3), label, fill="red")
    return image


def show_random_samples(dataset_root: Path, split: str = "train", count: int = 9, seed: int = 7) -> None:
    import random

    random.seed(seed)
    names = load_class_names(dataset_root)
    images = image_files(dataset_root / split / "images")
    if not images:
        print(f"No images found for {split}")
        return

    selected = random.sample(images, k=min(count, len(images)))
    columns = min(3, len(selected))
    rows = (len(selected) + columns - 1) // columns
    plt.figure(figsize=(5 * columns, 4 * rows))

    for index, image_path in enumerate(selected, start=1):
        label_path = dataset_root / split / "labels" / f"{image_path.stem}.txt"
        rendered = draw_yolo_boxes(image_path, label_path, names)
        plt.subplot(rows, columns, index)
        plt.imshow(rendered)
        plt.axis("off")
        plt.title(image_path.name[:40])

    plt.tight_layout()
    plt.show()


if final_dataset_root is not None:
    show_random_samples(final_dataset_root, split="train", count=9)
else:
    print("No final dataset to visualize yet.")


## Class balance

In [ ]:
def plot_class_balance(dataset_root: Path) -> None:
    summary = summarize_yolo_source(dataset_root)
    names = load_class_names(dataset_root)
    split_counts = {split: [summary[split][name] for name in names] for split in SPLITS}

    x_positions = range(len(names))
    bottom = [0] * len(names)
    plt.figure(figsize=(14, 5))
    for split in SPLITS:
        values = split_counts[split]
        plt.bar(x_positions, values, bottom=bottom, label=split)
        bottom = [current + value for current, value in zip(bottom, values)]

    plt.xticks(list(x_positions), names, rotation=45, ha="right")
    plt.ylabel("objects")
    plt.title("Canonical class balance")
    plt.legend()
    plt.tight_layout()
    plt.show()


if final_dataset_root is not None:
    plot_class_balance(final_dataset_root)
else:
    print("No final dataset to plot yet.")


## Export final dataset ZIP

This is the handoff artifact for the model-training notebook. The ZIP contains the balanced YOLO dataset folder with `data.yaml`, `train/`, `valid/`, and `test/`.


In [ ]:
# Export the final balanced YOLO dataset as a zip for the training notebook.
import json
import shutil
from pathlib import Path

DATASET_EXPORT_NAME = "chicken_threat_dataset_v3"
DATASET_EXPORT_ROOT = final_dataset_root if "final_dataset_root" in locals() and final_dataset_root is not None else None

if DATASET_EXPORT_ROOT is None or not Path(DATASET_EXPORT_ROOT).exists():
    raise RuntimeError("No final dataset found. Run the merge, balance, and validation cells first.")

DATASET_EXPORT_ROOT = Path(DATASET_EXPORT_ROOT).resolve()
if not (DATASET_EXPORT_ROOT / "data.yaml").exists():
    raise RuntimeError(f"data.yaml missing in final dataset: {DATASET_EXPORT_ROOT}")

# Save a small manifest next to data.yaml so the model notebook can verify provenance.
manifest = {
    "name": DATASET_EXPORT_NAME,
    "dataset_root_name": DATASET_EXPORT_ROOT.name,
    "canonical_names": CANONICAL_NAMES,
    "splits": SPLITS,
    "source_notebook": "ChickenThreadDetectorDatav3.ipynb",
}
(DATASET_EXPORT_ROOT / "dataset_manifest_v3.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_base = WORK_DIR / DATASET_EXPORT_NAME
zip_file = Path(shutil.make_archive(str(zip_base), "zip", root_dir=DATASET_EXPORT_ROOT))
print(f"Exported dataset zip: {zip_file}")
print(f"Zip size: {zip_file.stat().st_size / (1024 * 1024):.1f} MB")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_file))
else:
    print("Download/copy this ZIP into the model notebook runtime:")
    print(zip_file)
